# Day 2 — Silver Layer, Data Quality and Event Processing

This notebook is rebuilt for the **updated Healthcare Patient Journey dataset** and the Bronze tables created by the revised Day 1 notebook.

It processes six Day 1 Bronze tables:
- Patient encounters
- Lab test events
- Bed transfer events
- Patient master seed
- Department master seed
- Bed master seed

**Important design decisions**
- The updated source has `admission_ts` and `discharge_ts`; the old `event_timestamp`, `bill_amount`, `wait_minutes`, and status-CDC assumptions are removed.
- There is no status CDC feed in the updated dataset. Lab and bed-transfer feeds are processed as event datasets instead.
- Bronze remains unchanged. Casting, standardization, validation, deduplication and quarantine happen in Silver.
- Exact duplicate rows can be collapsed safely. If the same business key has conflicting rows and there is no trustworthy source sequence, all rows for that key are quarantined rather than choosing an arbitrary winner.
- Bed-transfer `from_bed_id` / `to_bed_id` values use different prefixes from `bed_master.bed_id` in the supplied sample. This notebook does **not** falsely join them to the bed master; it preserves them and reports the mapping limitation for the team to resolve.
- Optional discharge timestamps are allowed for encounters that may still be open. Invalid chronology is rejected.

## 1. Imports, catalog and table configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE_SCHEMA = "hospital_schema"   # Must match the Day 1 notebook
SILVER_SCHEMA = "silver_schema"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SILVER_SCHEMA}`")

# Day 1 Bronze tables — names must match Day1_Healthcare_Updated.ipynb
BRONZE_ENCOUNTERS = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_patient_encounters"
BRONZE_LAB_EVENTS = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_lab_test_events"
BRONZE_BED_TRANSFERS = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_bed_transfer_events"
BRONZE_PATIENT_MASTER = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_patient_master_seed"
BRONZE_DEPARTMENT_MASTER = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_department_master_seed"
BRONZE_BED_MASTER = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_bed_master_seed"

# Day 2 Silver targets
SILVER_ENCOUNTERS = f"{CATALOG}.{SILVER_SCHEMA}.silver_patient_encounters"
SILVER_LAB_EVENTS = f"{CATALOG}.{SILVER_SCHEMA}.silver_lab_test_events"
SILVER_BED_TRANSFERS = f"{CATALOG}.{SILVER_SCHEMA}.silver_bed_transfer_events"
SILVER_PATIENT_MASTER = f"{CATALOG}.{SILVER_SCHEMA}.silver_patient_master"
SILVER_DEPARTMENT_MASTER = f"{CATALOG}.{SILVER_SCHEMA}.silver_department_master"
SILVER_BED_MASTER = f"{CATALOG}.{SILVER_SCHEMA}.silver_bed_master"
SILVER_REJECTED = f"{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records"
SILVER_DQ_RULES = f"{CATALOG}.{SILVER_SCHEMA}.silver_dq_rule_catalogue"

print("Catalog:", CATALOG)
print("Bronze schema:", BRONZE_SCHEMA)
print("Silver schema:", SILVER_SCHEMA)

## 2. Confirm that all six Day 1 Bronze tables exist

In [0]:
bronze_table_names = [
    BRONZE_ENCOUNTERS, BRONZE_LAB_EVENTS, BRONZE_BED_TRANSFERS,
    BRONZE_PATIENT_MASTER, BRONZE_DEPARTMENT_MASTER, BRONZE_BED_MASTER
]

for table_name in bronze_table_names:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Bronze table not found: {table_name}. "
            "Run the updated Day 1 notebook first and verify BRONZE_SCHEMA."
        )
    print("FOUND:", table_name)

## 3. Read Bronze data and inspect source schemas

In [0]:
encounters_bronze = spark.table(BRONZE_ENCOUNTERS)
lab_bronze = spark.table(BRONZE_LAB_EVENTS)
transfers_bronze = spark.table(BRONZE_BED_TRANSFERS)
patients_bronze = spark.table(BRONZE_PATIENT_MASTER)
departments_bronze = spark.table(BRONZE_DEPARTMENT_MASTER)
beds_bronze = spark.table(BRONZE_BED_MASTER)

source_frames = [
    ("patient_encounters", encounters_bronze),
    ("lab_test_events", lab_bronze),
    ("bed_transfer_events", transfers_bronze),
    ("patient_master", patients_bronze),
    ("department_master", departments_bronze),
    ("bed_master", beds_bronze)
]

for source_name, source_df in source_frames:
    print(f"\\n{source_name}: {source_df.count():,} rows")
    source_df.printSchema()
    display(source_df.limit(5))

## 4. Create the Data Quality Rules Catalogue

In [0]:
dq_rule_rows = [
    ("ENC-01", "patient_encounters", "encounter_id", "Encounter business key is present", "ERROR", "REJECT"),
    ("ENC-02", "patient_encounters", "patient_id", "Patient key is present and exists in patient master", "ERROR", "REJECT"),
    ("ENC-03", "patient_encounters", "department_id", "Department key is present and exists in department master", "ERROR", "REJECT"),
    ("ENC-04", "patient_encounters", "admission_ts", "Admission timestamp is valid", "ERROR", "REJECT"),
    ("ENC-05", "patient_encounters", "discharge_ts", "Discharge is not earlier than admission when supplied", "ERROR", "REJECT"),
    ("ENC-06", "patient_encounters", "bed_id", "Non-null bed key exists in bed master", "ERROR", "REJECT"),
    ("ENC-07", "patient_encounters", "encounter_id", "Conflicting duplicate encounter keys are quarantined", "ERROR", "QUARANTINE"),
    ("LAB-01", "lab_test_events", "lab_event_id", "Lab event business key is present", "ERROR", "REJECT"),
    ("LAB-02", "lab_test_events", "encounter_id", "Encounter key exists in Silver encounters", "ERROR", "REJECT"),
    ("LAB-03", "lab_test_events", "patient_id", "Patient key exists in patient master", "ERROR", "REJECT"),
    ("LAB-04", "lab_test_events", "ordered_ts", "Order timestamp is valid", "ERROR", "REJECT"),
    ("LAB-05", "lab_test_events", "event timestamps", "Collected and result timestamps follow event chronology", "ERROR", "REJECT"),
    ("LAB-06", "lab_test_events", "department_id", "Department key exists in department master", "ERROR", "REJECT"),
    ("TRN-01", "bed_transfer_events", "transfer_id", "Transfer business key is present", "ERROR", "REJECT"),
    ("TRN-02", "bed_transfer_events", "encounter_id", "Encounter key exists in Silver encounters", "ERROR", "REJECT"),
    ("TRN-03", "bed_transfer_events", "patient_id", "Patient key exists in patient master", "ERROR", "REJECT"),
    ("TRN-04", "bed_transfer_events", "transfer_ts", "Transfer timestamp is valid", "ERROR", "REJECT"),
    ("TRN-05", "bed_transfer_events", "recorded_ts", "Recorded timestamp is valid when supplied", "ERROR", "REJECT"),
    ("TRN-06", "bed_transfer_events", "from_bed_id/to_bed_id", "Source and destination bed identifiers are present and different", "ERROR", "REJECT"),
    ("MST-01", "master_data", "business key", "Master business key is present", "ERROR", "REJECT"),
    ("MST-02", "master_data", "business key", "Conflicting duplicate master keys are quarantined", "ERROR", "QUARANTINE"),
    ("MST-03", "bed_master", "department_id", "Bed department key exists in department master", "ERROR", "REJECT"),
    ("MST-04", "master_data", "typed attributes", "Non-empty date/numeric source values parse successfully", "ERROR", "REJECT"),
    ("DUP-01", "all", "business key", "Exact duplicate valid rows are collapsed and counted in reconciliation", "WARNING", "DEDUPLICATE"),
    ("FLAG-01", "all", "optional attributes", "Missing optional attributes are retained and flagged", "WARNING", "FLAG")
]

dq_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("source_dataset", T.StringType(), False),
    T.StructField("affected_columns", T.StringType(), False),
    T.StructField("rule_description", T.StringType(), False),
    T.StructField("severity", T.StringType(), False),
    T.StructField("handling_action", T.StringType(), False)
])

dq_catalogue = spark.createDataFrame(dq_rule_rows, dq_schema).withColumn(
    "created_at", F.current_timestamp()
)

(dq_catalogue.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_DQ_RULES))

display(dq_catalogue.orderBy("rule_id"))
print("Saved DQ catalogue:", SILVER_DQ_RULES)

## 5. Standardization helpers — preserve raw values before casting

In [0]:
def require_columns(df, required_columns, dataset_name):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} is missing required columns: {missing}")

def add_raw_columns(df, columns):
    result = df
    for column_name in columns:
        if column_name in result.columns:
            result = result.withColumn(f"{column_name}_raw", F.col(column_name).cast("string"))
    return result

def add_source_hash(df):
    original_columns = [c for c in df.columns if not c.startswith("_")]
    return df.withColumn(
        "_source_row_hash",
        F.sha2(F.to_json(F.struct(*[F.col(c) for c in original_columns])), 256)
    )

def standardize_key(df, key_columns):
    result = df
    for column_name in key_columns:
        if column_name in result.columns:
            result = result.withColumn(
                column_name, F.trim(F.col(column_name).cast("string"))
            )
    return result

def make_rejects(df, dataset_name, key_column, rule_id, reason):
    # Store the full row as JSON so rejected source evidence remains traceable.
    return df.filter(F.col("_rule_id").isNotNull()).select(
        F.lit(dataset_name).alias("source_dataset"),
        F.col(key_column).cast("string").alias("business_key"),
        F.col("_rule_id").alias("rule_id"),
        F.col("_rejection_reason").alias("rejection_reason"),
        F.to_json(F.struct(*[F.col(c) for c in df.columns if c not in ["_rule_id", "_rejection_reason"]])).alias("source_record_json"),
        F.current_timestamp().alias("rejected_at")
    )

def exact_deduplicate(df):
    # Removes only fully identical rows. Conflicting business-key duplicates are handled separately.
    return df.dropDuplicates()

## 6. Prepare reference keys from Silver-ready master sources

In [0]:
# Normalize master keys for referential checks. The actual master Silver tables
# are created in Section 7 before dependent transaction/event tables are processed.
patient_keys = patients_bronze.select(
    F.trim(F.col("patient_id").cast("string")).alias("patient_id")
).filter(F.col("patient_id").isNotNull()).dropDuplicates()

department_keys = departments_bronze.select(
    F.trim(F.col("department_id").cast("string")).alias("department_id")
).filter(F.col("department_id").isNotNull()).dropDuplicates()

bed_keys = beds_bronze.select(
    F.trim(F.col("bed_id").cast("string")).alias("bed_id")
).filter(F.col("bed_id").isNotNull()).dropDuplicates()

print("Patient reference keys:", patient_keys.count())
print("Department reference keys:", department_keys.count())
print("Bed reference keys:", bed_keys.count())

## 7. Clean and write the three master Silver tables

In [0]:
master_reject_frames = []
master_exact_duplicate_counts = {}
master_reject_counts = {}

def clean_master(source_df, dataset_name, key_column, target_table, extra_timestamp_columns=None):
    required = [key_column]
    require_columns(source_df, required, dataset_name)

    raw_fields = extra_timestamp_columns or []
    if dataset_name == "department_master":
        raw_fields += ["floor", "capacity_beds"]
    elif dataset_name == "bed_master":
        raw_fields += ["floor"]
    df = add_raw_columns(source_df, raw_fields)
    df = standardize_key(df, [key_column])

    # Apply explicit types while retaining *_raw source values for traceability.
    if dataset_name == "patient_master" and "registration_date" in df.columns:
        df = df.withColumn("registration_date", F.to_date("registration_date"))
    elif dataset_name == "department_master":
        if "floor" in df.columns:
            df = df.withColumn("floor", F.col("floor").cast("int"))
        if "capacity_beds" in df.columns:
            df = df.withColumn("capacity_beds", F.col("capacity_beds").cast("int"))
    elif dataset_name == "bed_master":
        if "floor" in df.columns:
            df = df.withColumn("floor", F.col("floor").cast("int"))
        if "effective_date" in df.columns:
            df = df.withColumn("effective_date", F.to_date("effective_date"))

    df = add_source_hash(df)

    master_type_failure = F.lit(False)
    for typed_field in raw_fields:
        if typed_field in df.columns and f"{typed_field}_raw" in df.columns:
            parse_failed = (
                F.col(f"{typed_field}_raw").isNotNull() &
                (F.trim(F.col(f"{typed_field}_raw")) != "") &
                F.col(typed_field).isNull()
            )
            master_type_failure = master_type_failure | parse_failed

    df = df.withColumn(
        "_rule_id",
        F.when(F.col(key_column).isNull() | (F.col(key_column) == ""), F.lit("MST-01"))
         .when(master_type_failure, F.lit("MST-04"))
    ).withColumn(
        "_rejection_reason",
        F.when(F.col("_rule_id") == "MST-01", F.lit(f"Missing or blank {key_column}"))
         .when(F.col("_rule_id") == "MST-04", F.lit("Non-empty date/numeric value could not be parsed"))
    )

    invalid = make_rejects(df, dataset_name, key_column, None, None)
    valid = df.filter(F.col("_rule_id").isNull()).drop("_rule_id", "_rejection_reason")
    valid_before_exact_dedup = valid.count()
    valid = exact_deduplicate(valid)
    master_exact_duplicate_counts[dataset_name] = valid_before_exact_dedup - valid.count()
    master_reject_counts[dataset_name] = invalid.count()

    # If one key has multiple different records, quarantine every row for that key.
    duplicate_keys = (
        valid.groupBy(key_column).count().filter(F.col("count") > 1)
        .select(key_column)
    )
    conflicting = valid.join(duplicate_keys, key_column, "inner")
    unique_rows = valid.join(duplicate_keys, key_column, "left_anti")

    if dataset_name == "bed_master":
        # A bed must reference a known department. Do not silently retain orphan departments.
        unique_rows = unique_rows.join(
            department_keys.withColumn("_department_exists", F.lit(True)),
            "department_id", "left"
        ) if "department_id" in unique_rows.columns else unique_rows
        if "_department_exists" in unique_rows.columns:
            bad_department = unique_rows.filter(F.col("_department_exists").isNull())
            good_department = unique_rows.filter(F.col("_department_exists").isNotNull()).drop("_department_exists")
            if bad_department.limit(1).count():
                master_reject_counts[dataset_name] += bad_department.count()
                bad_department = (bad_department
                    .withColumn("_rule_id", F.lit("MST-03"))
                    .withColumn("_rejection_reason", F.lit("department_id not found in department master")))
                master_reject_frames.append(make_rejects(bad_department, dataset_name, key_column, None, None))
            unique_rows = good_department

    if conflicting.limit(1).count():
        master_reject_counts[dataset_name] += conflicting.count()
        conflicting = (conflicting
            .withColumn("_rule_id", F.lit("MST-02"))
            .withColumn("_rejection_reason", F.lit(f"Conflicting records share {key_column}; no source sequence available")))
        master_reject_frames.append(make_rejects(conflicting, dataset_name, key_column, None, None))

    if invalid.limit(1).count():
        master_reject_frames.append(invalid)

    unique_rows = unique_rows.drop("_source_row_hash")
    (unique_rows.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(target_table))

    print(dataset_name, "| source:", source_df.count(),
          "| Silver:", unique_rows.count(),
          "| conflicting key rows:", conflicting.count())
    return unique_rows

silver_patients = clean_master(
    patients_bronze, "patient_master", "patient_id", SILVER_PATIENT_MASTER,
    ["registration_date"]
)
silver_departments = clean_master(
    departments_bronze, "department_master", "department_id", SILVER_DEPARTMENT_MASTER
)

# Use validated Silver department keys when checking bed-master relationships.
department_keys = silver_departments.select("department_id").dropDuplicates()

silver_beds = clean_master(
    beds_bronze, "bed_master", "bed_id", SILVER_BED_MASTER,
    ["effective_date"]
)

# Downstream encounter/event checks must reference validated Silver master keys.
patient_keys = silver_patients.select("patient_id").dropDuplicates()
department_keys = silver_departments.select("department_id").dropDuplicates()
bed_keys = silver_beds.select("bed_id").dropDuplicates()

display(silver_patients.limit(5))
display(silver_departments.limit(5))
display(silver_beds.limit(5))

## 8. Validate and build the Silver patient encounters table

In [0]:
encounter_required = [
    "encounter_id", "patient_id", "department_id", "admission_ts",
    "discharge_ts", "encounter_type", "triage_priority", "encounter_status",
    "attending_clinician_id", "admission_source", "diagnosis_group", "bed_id"
]
require_columns(encounters_bronze, encounter_required, "patient_encounters")

enc = add_raw_columns(encounters_bronze, ["admission_ts", "discharge_ts"])
enc = standardize_key(enc, ["encounter_id", "patient_id", "department_id", "bed_id"])
enc = add_source_hash(enc)
enc = (enc
    .withColumn("admission_ts", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts", F.to_timestamp("discharge_ts"))
)

enc = (enc
    .join(patient_keys.withColumn("_patient_exists", F.lit(True)), "patient_id", "left")
    .join(department_keys.withColumn("_department_exists", F.lit(True)), "department_id", "left")
    .join(bed_keys.withColumn("_bed_exists", F.lit(True)), "bed_id", "left")
)

enc = (enc
    .withColumn("_rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "ENC-01")
         .when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "ENC-02")
         .when(F.col("department_id").isNull() | (F.col("department_id") == ""), "ENC-03")
         .when(F.col("admission_ts").isNull(), "ENC-04")
         .when(F.col("discharge_ts").isNotNull() & (F.col("discharge_ts") < F.col("admission_ts")), "ENC-05")
         .when(F.col("_patient_exists").isNull(), "ENC-02")
         .when(F.col("_department_exists").isNull(), "ENC-03")
         .when(F.col("bed_id").isNotNull() & (F.col("bed_id") != "") & F.col("_bed_exists").isNull(), "ENC-06")
    )
    .withColumn("_rejection_reason",
        F.when(F.col("_rule_id") == "ENC-01", "Missing encounter_id")
         .when(F.col("_rule_id") == "ENC-02", "Missing or unmatched patient_id")
         .when(F.col("_rule_id") == "ENC-03", "Missing or unmatched department_id")
         .when(F.col("_rule_id") == "ENC-04", "Missing or invalid admission_ts")
         .when(F.col("_rule_id") == "ENC-05", "discharge_ts is earlier than admission_ts")
         .when(F.col("_rule_id") == "ENC-06", "Non-null bed_id not found in bed master")
    )
)

encounter_rejects = make_rejects(enc, "patient_encounters", "encounter_id", None, None)
enc_valid = enc.filter(F.col("_rule_id").isNull()).drop(
    "_rule_id", "_rejection_reason", "_patient_exists", "_department_exists", "_bed_exists"
)
enc_valid_before_exact_dedup = enc_valid.count()
enc_valid = exact_deduplicate(enc_valid)
enc_exact_duplicate_extra = enc_valid_before_exact_dedup - enc_valid.count()

enc_duplicate_keys = (
    enc_valid.groupBy("encounter_id").count().filter(F.col("count") > 1)
    .select("encounter_id")
)
enc_conflicts = enc_valid.join(enc_duplicate_keys, "encounter_id", "inner")
enc_unique = enc_valid.join(enc_duplicate_keys, "encounter_id", "left_anti")

enc_conflict_rejects = (enc_conflicts
    .withColumn("_rule_id", F.lit("ENC-07"))
    .withColumn("_rejection_reason", F.lit("Conflicting records share encounter_id; no source sequence available")))
enc_conflict_rejects = make_rejects(enc_conflict_rejects, "patient_encounters", "encounter_id", None, None)

optional_encounter_fields = ["encounter_type", "triage_priority", "encounter_status",
                             "attending_clinician_id", "admission_source", "diagnosis_group", "bed_id"]
missing_optional = None
for field_name in optional_encounter_fields:
    expr = F.col(field_name).isNull() | (F.trim(F.col(field_name).cast("string")) == "")
    missing_optional = expr if missing_optional is None else (missing_optional | expr)

silver_encounters = (enc_unique
    .withColumn("missing_optional_attribute_flag", missing_optional)
    .withColumn("dq_status", F.when(F.col("missing_optional_attribute_flag"), "FLAGGED").otherwise("VALID"))
    .drop("_source_row_hash")
)

(silver_encounters.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_ENCOUNTERS))

print("Encounter Bronze rows:", encounters_bronze.count())
print("Encounter rows rejected by field/reference rules:", encounter_rejects.count())
print("Conflicting encounter-key rows quarantined:", enc_conflict_rejects.count())
print("Silver encounter rows:", silver_encounters.count())
display(silver_encounters.groupBy("dq_status").count())

## 9. Validate lab events — event timestamps and parent references

In [0]:
lab_required = [
    "lab_event_id", "encounter_id", "patient_id", "test_code",
    "ordered_ts", "collected_ts", "result_ts", "result_status",
    "department_id", "priority"
]
require_columns(lab_bronze, lab_required, "lab_test_events")

silver_encounter_keys = silver_encounters.select("encounter_id").dropDuplicates()
silver_patient_keys = silver_patients.select("patient_id").dropDuplicates()
silver_department_keys = silver_departments.select("department_id").dropDuplicates()

lab = add_raw_columns(lab_bronze, ["ordered_ts", "collected_ts", "result_ts"])
lab = standardize_key(lab, ["lab_event_id", "encounter_id", "patient_id", "department_id"])
lab = add_source_hash(lab)
lab = (lab
    .withColumn("ordered_ts", F.to_timestamp("ordered_ts"))
    .withColumn("collected_ts", F.to_timestamp("collected_ts"))
    .withColumn("result_ts", F.to_timestamp("result_ts"))
    .join(silver_encounter_keys.withColumn("_encounter_exists", F.lit(True)), "encounter_id", "left")
    .join(silver_patient_keys.withColumn("_patient_exists", F.lit(True)), "patient_id", "left")
    .join(silver_department_keys.withColumn("_department_exists", F.lit(True)), "department_id", "left")
)

lab = (lab
    .withColumn("_rule_id",
        F.when(F.col("lab_event_id").isNull() | (F.col("lab_event_id") == ""), "LAB-01")
         .when(F.col("encounter_id").isNull() | F.col("_encounter_exists").isNull(), "LAB-02")
         .when(F.col("patient_id").isNull() | F.col("_patient_exists").isNull(), "LAB-03")
         .when(F.col("ordered_ts").isNull(), "LAB-04")
         .when(F.col("collected_ts").isNotNull() & (F.col("collected_ts") < F.col("ordered_ts")), "LAB-05")
         .when(F.col("result_ts").isNotNull() & F.col("collected_ts").isNotNull() &
               (F.col("result_ts") < F.col("collected_ts")), "LAB-05")
         .when(F.col("result_ts").isNotNull() & F.col("collected_ts").isNull() &
               (F.col("result_ts") < F.col("ordered_ts")), "LAB-05")
         .when(F.col("department_id").isNull() | F.col("_department_exists").isNull(), "LAB-06")
    )
    .withColumn("_rejection_reason",
        F.when(F.col("_rule_id") == "LAB-01", "Missing lab_event_id")
         .when(F.col("_rule_id") == "LAB-02", "Missing or unmatched encounter_id in Silver encounters")
         .when(F.col("_rule_id") == "LAB-03", "Missing or unmatched patient_id")
         .when(F.col("_rule_id") == "LAB-04", "Missing or invalid ordered_ts")
         .when(F.col("_rule_id") == "LAB-05", "Lab event timestamps are out of chronological order")
         .when(F.col("_rule_id") == "LAB-06", "Missing or unmatched department_id")
    )
)

lab_rejects = make_rejects(lab, "lab_test_events", "lab_event_id", None, None)
lab_valid = lab.filter(F.col("_rule_id").isNull()).drop(
    "_rule_id", "_rejection_reason", "_encounter_exists", "_patient_exists", "_department_exists"
)
lab_valid_before_exact_dedup = lab_valid.count()
lab_valid = exact_deduplicate(lab_valid)
lab_exact_duplicate_extra = lab_valid_before_exact_dedup - lab_valid.count()
lab_duplicate_keys = lab_valid.groupBy("lab_event_id").count().filter(F.col("count") > 1).select("lab_event_id")
lab_conflicts = lab_valid.join(lab_duplicate_keys, "lab_event_id", "inner")
lab_unique = lab_valid.join(lab_duplicate_keys, "lab_event_id", "left_anti")
lab_conflict_rejects = (lab_conflicts
    .withColumn("_rule_id", F.lit("LAB-01"))
    .withColumn("_rejection_reason", F.lit("Conflicting rows share lab_event_id; no source sequence available")))
lab_conflict_rejects = make_rejects(lab_conflict_rejects, "lab_test_events", "lab_event_id", None, None)

lab_optional_fields = ["test_code", "result_status", "priority", "collected_ts", "result_ts"]
lab_missing_optional = None
for field_name in lab_optional_fields:
    expr = F.col(field_name).isNull() | (F.trim(F.col(field_name).cast("string")) == "")
    lab_missing_optional = expr if lab_missing_optional is None else (lab_missing_optional | expr)

silver_lab_events = (lab_unique
    .withColumn("missing_optional_attribute_flag", lab_missing_optional)
    .withColumn("dq_status", F.when(F.col("missing_optional_attribute_flag"), "FLAGGED").otherwise("VALID"))
    .drop("_source_row_hash")
)

(silver_lab_events.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_LAB_EVENTS))

print("Lab Bronze rows:", lab_bronze.count())
print("Lab rejected rows:", lab_rejects.count())
print("Lab conflicting-key rows:", lab_conflict_rejects.count())
print("Silver lab rows:", silver_lab_events.count())
display(silver_lab_events.groupBy("result_status", "dq_status").count())

## 10. Validate bed-transfer events — lifecycle and event ordering

In [0]:
transfer_required = [
    "transfer_id", "encounter_id", "patient_id", "from_bed_id", "to_bed_id",
    "transfer_ts", "transfer_reason", "department_id", "transfer_status", "recorded_ts"
]
require_columns(transfers_bronze, transfer_required, "bed_transfer_events")

trans = add_raw_columns(transfers_bronze, ["transfer_ts", "recorded_ts"])
trans = standardize_key(trans, ["transfer_id", "encounter_id", "patient_id", "from_bed_id", "to_bed_id", "department_id"])
trans = add_source_hash(trans)
trans = (trans
    .withColumn("transfer_ts", F.to_timestamp("transfer_ts"))
    .withColumn("recorded_ts", F.to_timestamp("recorded_ts"))
    .join(silver_encounter_keys.withColumn("_encounter_exists", F.lit(True)), "encounter_id", "left")
    .join(silver_patient_keys.withColumn("_patient_exists", F.lit(True)), "patient_id", "left")
    .join(silver_department_keys.withColumn("_department_exists", F.lit(True)), "department_id", "left")
)

trans = (trans
    .withColumn("_rule_id",
        F.when(F.col("transfer_id").isNull() | (F.col("transfer_id") == ""), "TRN-01")
         .when(F.col("encounter_id").isNull() | F.col("_encounter_exists").isNull(), "TRN-02")
         .when(F.col("patient_id").isNull() | F.col("_patient_exists").isNull(), "TRN-03")
         .when(F.col("transfer_ts").isNull(), "TRN-04")
         .when(F.col("recorded_ts_raw").isNotNull() & F.col("recorded_ts").isNull(), "TRN-05")
         .when(F.col("from_bed_id").isNull() | (F.col("from_bed_id") == "") |
               F.col("to_bed_id").isNull() | (F.col("to_bed_id") == "") |
               (F.col("from_bed_id") == F.col("to_bed_id")), "TRN-06")
         .when(F.col("department_id").isNull() | F.col("_department_exists").isNull(), "TRN-03")
    )
    .withColumn("_rejection_reason",
        F.when(F.col("_rule_id") == "TRN-01", "Missing transfer_id")
         .when(F.col("_rule_id") == "TRN-02", "Missing or unmatched encounter_id")
         .when(F.col("_rule_id") == "TRN-03", "Missing or unmatched patient_id or department_id")
         .when(F.col("_rule_id") == "TRN-04", "Missing or invalid transfer_ts")
         .when(F.col("_rule_id") == "TRN-05", "recorded_ts supplied but could not be parsed")
         .when(F.col("_rule_id") == "TRN-06", "from_bed_id and to_bed_id must both be present and different")
    )
)

transfer_rejects = make_rejects(trans, "bed_transfer_events", "transfer_id", None, None)
trans_valid = trans.filter(F.col("_rule_id").isNull()).drop(
    "_rule_id", "_rejection_reason", "_encounter_exists", "_patient_exists", "_department_exists"
)
trans_valid_before_exact_dedup = trans_valid.count()
trans_valid = exact_deduplicate(trans_valid)
transfer_exact_duplicate_extra = trans_valid_before_exact_dedup - trans_valid.count()
transfer_duplicate_keys = trans_valid.groupBy("transfer_id").count().filter(F.col("count") > 1).select("transfer_id")
transfer_conflicts = trans_valid.join(transfer_duplicate_keys, "transfer_id", "inner")
transfer_unique = trans_valid.join(transfer_duplicate_keys, "transfer_id", "left_anti")
transfer_conflict_rejects = (transfer_conflicts
    .withColumn("_rule_id", F.lit("TRN-01"))
    .withColumn("_rejection_reason", F.lit("Conflicting rows share transfer_id; no source sequence available")))
transfer_conflict_rejects = make_rejects(transfer_conflict_rejects, "bed_transfer_events", "transfer_id", None, None)

transfer_optional_fields = ["transfer_reason", "transfer_status", "recorded_ts"]
transfer_missing_optional = None
for field_name in transfer_optional_fields:
    expr = F.col(field_name).isNull() | (F.trim(F.col(field_name).cast("string")) == "")
    transfer_missing_optional = expr if transfer_missing_optional is None else (transfer_missing_optional | expr)

silver_bed_transfers = (transfer_unique
    .withColumn("missing_optional_attribute_flag", transfer_missing_optional)
    .withColumn("dq_status", F.when(F.col("missing_optional_attribute_flag"), "FLAGGED").otherwise("VALID"))
    .drop("_source_row_hash")
)

(silver_bed_transfers.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_BED_TRANSFERS))

print("Transfer Bronze rows:", transfers_bronze.count())
print("Transfer rejected rows:", transfer_rejects.count())
print("Transfer conflicting-key rows:", transfer_conflict_rejects.count())
print("Silver transfer rows:", silver_bed_transfers.count())
display(silver_bed_transfers.groupBy("transfer_status", "dq_status").count())

## 11. Bed identifier mapping investigation — do not assume a false foreign key

In [0]:
# The source sample uses identifiers such as FROM_BE... and TO_BED...,
# while bed_master uses BED... . Measure the overlap before defining a join.
bed_master_ids = silver_beds.select(F.col("bed_id").alias("master_bed_id")).dropDuplicates()

from_bed_overlap = (
    silver_bed_transfers.select("from_bed_id").distinct()
    .join(bed_master_ids, F.col("from_bed_id") == F.col("master_bed_id"), "inner")
    .count()
)
to_bed_overlap = (
    silver_bed_transfers.select("to_bed_id").distinct()
    .join(bed_master_ids, F.col("to_bed_id") == F.col("master_bed_id"), "inner")
    .count()
)

print("Distinct transfer from_bed_id values matching bed master:", from_bed_overlap)
print("Distinct transfer to_bed_id values matching bed master:", to_bed_overlap)
print("Transfer source identifiers are retained as received.")
print("Do not join these identifiers to bed_master until the source owner supplies an approved crosswalk.")
display(silver_bed_transfers.select("from_bed_id", "to_bed_id").distinct().limit(20))

## 12. Combine all rejected records into one auditable quarantine table

In [0]:
reject_frames = [
    encounter_rejects, enc_conflict_rejects,
    lab_rejects, lab_conflict_rejects,
    transfer_rejects, transfer_conflict_rejects
] + master_reject_frames

if reject_frames:
    rejected_all = reject_frames[0]
    for reject_df in reject_frames[1:]:
        rejected_all = rejected_all.unionByName(reject_df, allowMissingColumns=True)
else:
    rejected_schema = T.StructType([
        T.StructField("source_dataset", T.StringType(), True),
        T.StructField("business_key", T.StringType(), True),
        T.StructField("rule_id", T.StringType(), True),
        T.StructField("rejection_reason", T.StringType(), True),
        T.StructField("source_record_json", T.StringType(), True),
        T.StructField("rejected_at", T.TimestampType(), True)
    ])
    rejected_all = spark.createDataFrame([], rejected_schema)

(rejected_all.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_REJECTED))

print("Saved rejected-record audit:", SILVER_REJECTED)
print("Total quarantined rows:", rejected_all.count())
display(rejected_all.groupBy("source_dataset", "rule_id").count().orderBy("source_dataset", "rule_id"))

## 13. Event-time and encounter-lifecycle checks

In [0]:
# Compare event timestamps with the encounter admission/discharge window.
# These checks are reported separately so the team can decide whether they are
# hard rejects or business exceptions after confirming source semantics.

encounter_window = silver_encounters.select(
    "encounter_id", "admission_ts", "discharge_ts"
).dropDuplicates(["encounter_id"])

lab_vs_encounter = (
    silver_lab_events.alias("l")
    .join(encounter_window.alias("e"), "encounter_id", "left")
    .withColumn("lab_before_admission",
        F.col("ordered_ts").isNotNull() & (F.col("ordered_ts") < F.col("admission_ts")))
    .withColumn("lab_after_discharge",
        F.col("discharge_ts").isNotNull() & F.col("result_ts").isNotNull() &
        (F.col("result_ts") > F.col("discharge_ts")))
)

transfer_vs_encounter = (
    silver_bed_transfers.alias("t")
    .join(encounter_window.alias("e"), "encounter_id", "left")
    .withColumn("transfer_before_admission",
        F.col("transfer_ts") < F.col("admission_ts"))
    .withColumn("transfer_after_discharge",
        F.col("discharge_ts").isNotNull() & (F.col("transfer_ts") > F.col("discharge_ts")))
)

print("Lab events ordered before encounter admission:",
      lab_vs_encounter.filter("lab_before_admission").count())
print("Lab results after encounter discharge:",
      lab_vs_encounter.filter("lab_after_discharge").count())
print("Bed transfers before encounter admission:",
      transfer_vs_encounter.filter("transfer_before_admission").count())
print("Bed transfers after encounter discharge:",
      transfer_vs_encounter.filter("transfer_after_discharge").count())

display(lab_vs_encounter.filter("lab_before_admission OR lab_after_discharge").select(
    "lab_event_id", "encounter_id", "ordered_ts", "result_ts", "admission_ts", "discharge_ts"
).limit(50))
display(transfer_vs_encounter.filter("transfer_before_admission OR transfer_after_discharge").select(
    "transfer_id", "encounter_id", "transfer_ts", "admission_ts", "discharge_ts"
).limit(50))

## 14. Build encounter stay duration as a validated Silver derived column

In [0]:
# A duration is computed only when both timestamps exist and chronology is valid.
# Open encounters keep a null duration; do not substitute the current time.
silver_encounters_with_duration = (
    silver_encounters
    .withColumn(
        "length_of_stay_minutes",
        F.when(
            F.col("admission_ts").isNotNull() &
            F.col("discharge_ts").isNotNull() &
            (F.col("discharge_ts") >= F.col("admission_ts")),
            (F.col("discharge_ts").cast("long") - F.col("admission_ts").cast("long")) / 60.0
        )
    )
)

(silver_encounters_with_duration.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_ENCOUNTERS))

display(silver_encounters_with_duration.select(
    "encounter_id", "encounter_type", "encounter_status",
    "admission_ts", "discharge_ts", "length_of_stay_minutes"
).limit(20))

## 15. Create event histories ordered by business event time

In [0]:
lab_event_history = silver_lab_events.orderBy("encounter_id", "ordered_ts", "lab_event_id")
transfer_event_history = silver_bed_transfers.orderBy("encounter_id", "transfer_ts", "transfer_id")

print("Lab events ordered by event time:")
display(lab_event_history.select(
    "encounter_id", "lab_event_id", "test_code", "ordered_ts",
    "collected_ts", "result_ts", "result_status"
).limit(30))

print("Bed transfers ordered by event time:")
display(transfer_event_history.select(
    "encounter_id", "transfer_id", "from_bed_id", "to_bed_id",
    "transfer_ts", "recorded_ts", "transfer_status"
).limit(30))

## 16. Silver reconciliation — source, accepted, quarantined and exact duplicates

In [0]:
def count_rows(df):
    return int(df.count())

reconciliation_rows = [
    ("patient_encounters", count_rows(encounters_bronze), count_rows(silver_encounters_with_duration),
     count_rows(encounter_rejects) + count_rows(enc_conflict_rejects), enc_exact_duplicate_extra),
    ("lab_test_events", count_rows(lab_bronze), count_rows(silver_lab_events),
     count_rows(lab_rejects) + count_rows(lab_conflict_rejects), lab_exact_duplicate_extra),
    ("bed_transfer_events", count_rows(transfers_bronze), count_rows(silver_bed_transfers),
     count_rows(transfer_rejects) + count_rows(transfer_conflict_rejects), transfer_exact_duplicate_extra),
    ("patient_master", count_rows(patients_bronze), count_rows(silver_patients),
     master_reject_counts.get("patient_master", 0), master_exact_duplicate_counts.get("patient_master", 0)),
    ("department_master", count_rows(departments_bronze), count_rows(silver_departments),
     master_reject_counts.get("department_master", 0), master_exact_duplicate_counts.get("department_master", 0)),
    ("bed_master", count_rows(beds_bronze), count_rows(silver_beds),
     master_reject_counts.get("bed_master", 0), master_exact_duplicate_counts.get("bed_master", 0))
]

reconciliation_df = spark.createDataFrame(
    reconciliation_rows,
    ["source_dataset", "bronze_rows", "silver_rows", "quarantined_rows", "exact_duplicate_rows_removed"]
).withColumn(
    "accounted_rows",
    F.col("silver_rows") + F.col("quarantined_rows") + F.col("exact_duplicate_rows_removed")
).withColumn(
    "unaccounted_rows", F.col("bronze_rows") - F.col("accounted_rows")
).withColumn(
    "reconciliation_status",
    F.when(F.col("unaccounted_rows") == 0, "PASS").otherwise("CHECK")
)

display(reconciliation_df)

## 17. Data-quality summary by rule and source

In [0]:
display(
    spark.table(SILVER_REJECTED)
    .groupBy("source_dataset", "rule_id", "rejection_reason")
    .count()
    .orderBy("source_dataset", "rule_id")
)

for table_name in [
    SILVER_PATIENT_MASTER, SILVER_DEPARTMENT_MASTER, SILVER_BED_MASTER,
    SILVER_ENCOUNTERS, SILVER_LAB_EVENTS, SILVER_BED_TRANSFERS
]:
    print(f"\\n{table_name}")
    df = spark.table(table_name)
    print("Rows:", df.count())
    print("Columns:", len(df.columns))

## 18. Basic Silver validation tests

In [0]:
# These are assertions on structural and key invariants, not hard-coded dataset defect counts.
assert spark.table(SILVER_ENCOUNTERS).filter(
    F.col("encounter_id").isNull() | F.col("patient_id").isNull() |
    F.col("department_id").isNull() | F.col("admission_ts").isNull()
).count() == 0, "Silver encounter mandatory-field test failed"

assert spark.table(SILVER_LAB_EVENTS).filter(
    F.col("lab_event_id").isNull() | F.col("encounter_id").isNull() |
    F.col("ordered_ts").isNull()
).count() == 0, "Silver lab mandatory-field test failed"

assert spark.table(SILVER_BED_TRANSFERS).filter(
    F.col("transfer_id").isNull() | F.col("encounter_id").isNull() |
    F.col("transfer_ts").isNull()
).count() == 0, "Silver transfer mandatory-field test failed"

assert spark.table(SILVER_ENCOUNTERS).groupBy("encounter_id").count().filter(
    F.col("count") > 1
).count() == 0, "Duplicate encounter business keys remain in Silver"

assert spark.table(SILVER_LAB_EVENTS).groupBy("lab_event_id").count().filter(
    F.col("count") > 1
).count() == 0, "Duplicate lab event keys remain in Silver"

assert spark.table(SILVER_BED_TRANSFERS).groupBy("transfer_id").count().filter(
    F.col("count") > 1
).count() == 0, "Duplicate transfer keys remain in Silver"

print("PASS: mandatory Silver fields")
print("PASS: unique Silver business keys")
print("PASS: Silver layer structural tests")

## 19. Final Day 2 handoff

In [0]:
final_tables = [
    SILVER_PATIENT_MASTER, SILVER_DEPARTMENT_MASTER, SILVER_BED_MASTER,
    SILVER_ENCOUNTERS, SILVER_LAB_EVENTS, SILVER_BED_TRANSFERS,
    SILVER_REJECTED, SILVER_DQ_RULES
]

final_summary = []
for table_name in final_tables:
    row_count = spark.table(table_name).count()
    final_summary.append((table_name, row_count, "CREATED"))

display(spark.createDataFrame(
    final_summary, ["silver_output_table", "row_count", "status"]
))

print("DAY 2 COMPLETE")
print("Bronze source tables were read only; no Bronze rows were modified.")
print("Silver tables, DQ rule catalogue and rejected-record audit have been written.")
print("Day 3 can now build encounter, lab and bed-transfer facts from these Silver outputs.")
print("Evidence to capture: DQ catalogue, rejected-record counts, Silver table schemas, event chronology checks and reconciliation.")